# Preprocessing for the supervised transformer

This notebook produces the training file consumed by `transformers_model_finetuner.py`.
It takes the fine-grained labels in `labeled.csv` and writes a **balanced, 20-class**
dataset to `labeled_balanced_20.csv`.

Two deliberate choices, both required for the model to reach the target taxonomy:

1. **No category merging.** Unlike `dataset_balancing.ipynb` (which merges the fine
   classes into a coarse 15-class set via `class_mapping`), we keep every original
   class so the model can predict the full taxonomy.
2. **Drop `NoN` and `unknown`.** `unknown` is not a real category — the model exists
   precisely to assign the unknown events to real classes — so training on it would
   teach the model to emit `unknown`. Dropping it yields exactly 20 classes.

The finetuner derives its class list (and classifier head size) directly from the
unique `class` values in this file, so this file fully defines the label space.

In [1]:
import pandas as pd

## 1. Load the labeled data

We only need the label and the cleaned text. `low_memory=False` avoids the mixed-type
warning on the wide source CSV.

In [2]:
INPUT_CSV = '../../data/labeled.csv'
OUTPUT_CSV = '../../data/labeled_balanced_20.csv'

dataset = pd.read_csv(INPUT_CSV, low_memory=False)
dataset = dataset[['class', 'clean_notes']]
print(f'Loaded {len(dataset)} rows from {INPUT_CSV}')

Loaded 214232 rows from ../../data/labeled.csv


## 2. Drop non-categories and empty text

`NoN` is a placeholder and `unknown` is not a class we want the model to emit. Rows
without `clean_notes` carry no signal and would break tokenization.

In [3]:
dataset = dataset[~dataset['class'].isin(['NoN', 'unknown'])]
dataset = dataset.dropna(subset=['clean_notes'])

print(f'{len(dataset)} rows across {dataset["class"].nunique()} classes after filtering\n')
dataset['class'].value_counts()

130311 rows across 20 classes after filtering



class
labor rights                 22563
pandemic                     20710
farmers                      12907
palestine-israel conflict    11996
education                     9366
women rights                  7597
ukraine-russia war            6493
climate                       5989
environment                   5630
policies & politics           5201
health care                   4556
lgbtq                         3303
public services               3235
immigration                   2645
discrimination                2531
animal welfare                2024
housing                       1032
blm                            926
culture                        806
unjust law enforcement         801
Name: count, dtype: int64

## 3. Balance the classes

Cap each class to at most `CAP_PER_CLASS` rows. This undersamples the dominant classes
(e.g. `labor rights`, `pandemic`) while leaving the smaller ones at their natural size —
it does not oversample. Adjust `CAP_PER_CLASS` to trade dataset size for balance.

In [4]:
CAP_PER_CLASS = 3000

def keep_first_n(df, n):
    return df.groupby('class').head(n)

dataset = keep_first_n(dataset, CAP_PER_CLASS)

print(f'{len(dataset)} rows across {dataset["class"].nunique()} classes after balancing\n')
dataset['class'].value_counts()

49765 rows across 20 classes after balancing



class
environment                  3000
labor rights                 3000
policies & politics          3000
education                    3000
public services              3000
farmers                      3000
women rights                 3000
ukraine-russia war           3000
health care                  3000
palestine-israel conflict    3000
lgbtq                        3000
climate                      3000
pandemic                     3000
immigration                  2645
discrimination               2531
animal welfare               2024
housing                      1032
blm                           926
culture                       806
unjust law enforcement        801
Name: count, dtype: int64

## 4. (Optional) Inspect the class distribution

In [5]:
import plotly.express as px
from plotly.offline import iplot, init_notebook_mode

init_notebook_mode(True)

counts = dataset['class'].value_counts()
fig = px.bar(x=counts.index, y=counts.values, color=counts.index, text=counts.values)
fig.update_traces(hovertemplate="Category:'%{x}' Counted: %{y}")
fig.update_layout(
    title={'text': 'Balanced Category Counts', 'x': 0.5, 'font': {'size': 30}},
    xaxis={'title': 'Category', 'showgrid': False},
    yaxis={'title': 'Count', 'showgrid': False},
    plot_bgcolor='white', width=900, height=500, showlegend=False,
)
iplot(fig)

## 5. Write the training file

This is the file `transformers_model_finetuner.py` (`file_path`) and
`labeling_by_transformer_model.ipynb` (`TRAIN_CSV`) read.

In [6]:
dataset.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(dataset)} rows, {dataset["class"].nunique()} classes to {OUTPUT_CSV}')
print('Classes:', sorted(dataset['class'].unique()))

Wrote 49765 rows, 20 classes to ../../data/labeled_balanced_20.csv
Classes: ['animal welfare', 'blm', 'climate', 'culture', 'discrimination', 'education', 'environment', 'farmers', 'health care', 'housing', 'immigration', 'labor rights', 'lgbtq', 'palestine-israel conflict', 'pandemic', 'policies & politics', 'public services', 'ukraine-russia war', 'unjust law enforcement', 'women rights']
